# `long` / `wide` — Advanced Reference

`long` melts all numeric columns to rows. `wide` pivots a column's values into headers.

| Method | Key params | Purpose |
|---|---|---|
| `.long(col='variable', value='value')` | `col`, `value` | wide → long (melt) |
| `.wide(col='variable', value='value', aggfunc=None)` | `col`, `value`, `aggfunc` | long → wide (pivot) |

They are designed to work together as a pair, but each is also useful standalone.

---

In [ ]:
import sys, os
_src = os.path.abspath(os.path.join(os.getcwd(), '..'))
if _src not in sys.path: sys.path.insert(0, _src)

import pytae as pt

penguins = pt.sample_data['penguins']
healthexp = pt.sample_data['healthexp']
flights = pt.sample_data['flights']

## `long` — melt all numeric columns to rows

In [2]:
# Default names: 'variable' and 'value'
(penguins
 .agg_df(aggfunc='mean')
 .long()
 .head(10)
)

,species,island,sex,variable,value
0,Adelie,Biscoe,Female,bill_length_mm,37.359091
1,Adelie,Biscoe,Male,bill_length_mm,40.590909
2,Adelie,Dream,Female,bill_length_mm,36.911111
3,Adelie,Dream,Male,bill_length_mm,40.071429
4,Adelie,Torgersen,Female,bill_length_mm,37.554167
5,Adelie,Torgersen,Male,bill_length_mm,40.586957
6,Chinstrap,Dream,Female,bill_length_mm,46.573529
7,Chinstrap,Dream,Male,bill_length_mm,51.094118
8,Gentoo,Biscoe,Female,bill_length_mm,45.563793
9,Gentoo,Biscoe,Male,bill_length_mm,49.473770


In [ ]:
# group_x before long: add group context columns BEFORE melting, so they survive the melt
# Here: add species count per island group, then go long to see metrics alongside group size
(penguins
 .pipe(pt.group_x, group=['species', 'island'])  # adds 'n' column per group
 .agg_df(aggfunc='mean')
 .long(col='metric', value='mean_value')
 .head(12)
)

In [3]:
# Custom column names for the variable and value columns
(penguins
 .agg_df(aggfunc='mean')
 .long(col='metric', value='mean_value')
 .head(10)
)

,species,island,sex,metric,mean_value
0,Adelie,Biscoe,Female,bill_length_mm,37.359091
1,Adelie,Biscoe,Male,bill_length_mm,40.590909
2,Adelie,Dream,Female,bill_length_mm,36.911111
3,Adelie,Dream,Male,bill_length_mm,40.071429
4,Adelie,Torgersen,Female,bill_length_mm,37.554167
5,Adelie,Torgersen,Male,bill_length_mm,40.586957
6,Chinstrap,Dream,Female,bill_length_mm,46.573529
7,Chinstrap,Dream,Male,bill_length_mm,51.094118
8,Gentoo,Biscoe,Female,bill_length_mm,45.563793
9,Gentoo,Biscoe,Male,bill_length_mm,49.473770


## `wide` — pivot a column's values into headers

In [4]:
# healthexp: mean spending per country per year → pivot countries to columns
(healthexp
 .agg_df(aggfunc='mean')
 .wide(col='Country', value='Spending_USD')
 .head()
)

,Year,Life_Expectancy,Canada,France,Germany,Great Britain,Japan,USA
0,1995.000000,75.843137,NaN,NaN,NaN,NaN,NaN,4388.570529
1,1995.000000,79.554902,NaN,NaN,NaN,NaN,1860.257902,NaN
2,1995.080000,76.726000,NaN,NaN,2667.2802,NaN,NaN,NaN
3,1998.318182,78.706818,2685.778341,NaN,NaN,NaN,NaN,NaN
4,1998.627907,77.620930,NaN,NaN,NaN,2034.192465,NaN,NaN


In [ ]:
# wide with aggfunc — handles duplicates (multiple rows per group) via pivot_table
# note: penguins dataset has multiple rows for the same species/island group,
# so aggfunc is required when pivoting raw penguins data
(flights
 .wide(col='month', value='passengers', aggfunc='sum')
 .head()
)

,year,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
0,1949,112,118,132,129,121,135,148,148,136,119,104,118
1,1950,115,126,141,135,125,149,170,170,158,133,114,140
2,1951,145,150,178,163,172,178,199,199,184,162,146,166
3,1952,171,180,193,181,183,218,230,242,209,191,172,194
4,1953,196,196,236,235,229,243,264,272,237,211,180,201


## Round-trip: `long` → transform → `wide`
A common pattern: go long to apply uniform operations across all metrics, then come back wide.

In [6]:
# Aggregate penguins → go long → filter to bill metrics only → pivot species to columns
(penguins
 .agg_df(aggfunc='mean')
 .long(col='metric', value='mean_value')
 .qry({'metric': ['bill_length_mm', 'bill_depth_mm']})
 .wide(col='species', value='mean_value')
)

,island,sex,metric,Adelie,Chinstrap,Gentoo
0,Biscoe,Female,bill_depth_mm,17.704545,NaN,14.237931
1,Biscoe,Female,bill_length_mm,37.359091,NaN,45.563793
2,Biscoe,Male,bill_depth_mm,19.036364,NaN,15.718033
3,Biscoe,Male,bill_length_mm,40.590909,NaN,49.473770
4,Dream,Female,bill_depth_mm,17.618519,17.588235,NaN
5,Dream,Female,bill_length_mm,36.911111,46.573529,NaN
6,Dream,Male,bill_depth_mm,18.839286,19.252941,NaN
7,Dream,Male,bill_length_mm,40.071429,51.094118,NaN
8,Torgersen,Female,bill_depth_mm,17.550000,NaN,NaN
9,Torgersen,Female,bill_length_mm,37.554167,NaN,NaN
